# RoboBarista — Kameras, AprilTags und Caliscope im Direktvergleich

**16 Versuchskombinationen:** 1 oder 2 seitliche AprilTags × 3 oder 4 Kameras × OpenCV oder offizielles AprilTag 3 × **zwei Rekonstruktionswege: unsere 2D-Optimierung oder Caliscope 3D-Triangulation**.

Caliscope nutzt hier **seine echte, veröffentlichte Triangulations-API**, nicht eine von uns nachgebildete Funktion. Aus den triangulierten 3D-Tag-Ecken berechnen wir anschließend die starre Kännchen-Pose. Damit Caliscope mit demselben Setup verglichen werden kann, werden beide Verfahren mit den **exakt bekannten MuJoCo-Kameras** versorgt.

**Wichtige Einschränkung:** Das ist **noch kein Test von Caliscopes Kamera-Kalibrierungsfunktion**, weil die simulierten Kameras bereits perfekt kalibriert sind. Caliscope-Kalibrierung mit echten oder künstlich verrauschten Kamerabildern folgt als separate spätere Studie. Bei nur einem sichtbaren Kamera-Tag kann Caliscope nicht triangulieren.

Pro Variante laufen dieselben drei statischen Testposen und Neigungen mit 1280×960 Pixeln und festem Zufalls-Seed. **Forschungsziele:** höchstens 0,5 mm absolute Ausgießerabweichung und 1° Kännchenwinkel; relative Spitzenposition zur Tasse wird zusätzlich ausgewiesen.

**Keine GPU nötig:** Colab → Laufzeit → Laufzeittyp ändern → Hardwarebeschleuniger „Keine“. Danach **Laufzeit → Alle ausführen**. Die letzte Zelle zeigt die Vergleichstabelle und einen ZIP-Export. Verlorene Erkennungen und Abstürze gelten als Fehler, nicht als Erfolg.


In [ ]:
# 1. GitHub-Projekt laden (keine ZIP nötig)
import os, subprocess, sys, pathlib
repo = pathlib.Path('/content/robobarista-vision')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Coffee-Tobi/robocasa.git', str(repo)], check=True)
else:
    # Bei erneutem Ausführen aktuelle Verbesserungen von GitHub holen.
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
print('Projekt:', repo)


In [ ]:
# 2. Systemgrafik (CPU/OSMesa) und Python-Pakete installieren
# Ohne GPU benötigt MuJoCo echte OSMesa-Systembibliotheken; pip allein reicht nicht.
import os, subprocess, sys, ctypes.util
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'libosmesa6-dev',
                'libgl1', 'libglx-mesa0', 'libgl1-mesa-dri'], check=True)
osmesa_lib = ctypes.util.find_library('OSMesa')
print('OSMesa-Systembibliothek:', osmesa_lib)
if not osmesa_lib:
    raise RuntimeError('OSMesa fehlt trotz apt-Installation. Bitte die Ausgabe von Schritt 2 teilen.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'requirements.txt'], check=True)
# Caliscope ist ein optionales Forschungswerkzeug in diesem Vergleichsnotebook.
# Pure Python-API, kein Desktop-GUI nötig.
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'caliscope==0.11.8'], check=True)
from caliscope.api import CameraData, CameraArray, ImagePoints
print('Caliscope API: verfügbar')

# Eigenständiger Importcheck im richtigen Rendering-Backend (ohne GPU: OSMesa).
backend = 'egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
check_env = os.environ.copy()
check_env['MUJOCO_GL'] = backend
if backend == 'osmesa':
    check_env['PYOPENGL_PLATFORM'] = 'osmesa'
print('MuJoCo-Importtest mit Backend:', backend, flush=True)
subprocess.run([sys.executable, '-c',
                'import mujoco; print("MuJoCo:", mujoco.__version__, "Import OK")'],
               env=check_env, check=True)
import cv2, numpy as np
print('OpenCV:', cv2.__version__, 'NumPy:', np.__version__)


In [ ]:
# 3. Mathematische Tests – vollständige Fehleranzeige statt kryptischem Exit-Code
import subprocess,sys
test = subprocess.run(
    [sys.executable, '-m', 'pytest', '-vv', '-ra', '--tb=short', 'tests'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, errors='replace', check=False)
print('=== Testergebnisse (stdout) ===', flush=True)
print(test.stdout[-30000:] if test.stdout.strip() else '(leer)', flush=True)
print('=== Zusätzliche Fehlermeldungen (stderr) ===', flush=True)
print(test.stderr[-20000:] if test.stderr.strip() else '(leer)', flush=True)
if test.returncode:
    raise RuntimeError(
        'Mindestens ein Python-Test ist fehlgeschlagen. '
        'Oben steht der genaue Testname unter FAILURES/ERRORS; '
        'bitte diesen Abschnitt teilen. Nicht mit unvalidierten Messungen fortfahren.')
print('Alle Python-Tests bestanden.')


In [ ]:
# 4. 16 Varianten unter gleichen Bedingungen durchrechnen (kein GPU-Zwang)
import os,sys,subprocess
env=os.environ.copy()
env['MUJOCO_GL']='egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
if env['MUJOCO_GL']=='osmesa':
    env['PYOPENGL_PLATFORM']='osmesa'
print("Rendering:",env['MUJOCO_GL'],flush=True)
cmd=[sys.executable,'-u','-m','robobarista.compare_rigs',
     '--trials','3','--width','1280','--height','960']
result=subprocess.run(cmd,cwd=str(repo),env=env,
    stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True,errors='replace')
print(result.stdout if result.stdout else '(Keine Standardausgabe)')
print('=== Fehlerhinweise ===')
print(result.stderr[-12000:] if result.stderr else '(keine)')
print('Gesamt-Exit-Code:',result.returncode)


In [ ]:
# 5. Vergleichstabelle, 4-Kamera-Montage und ZIP zum Sichern
from pathlib import Path
import json, pandas as pd
from IPython.display import display, Image, FileLink
import zipfile
folder=Path(repo)/'runs/rig_comparison'
p=folder/'comparison.json'
if not p.exists():
    print('Noch kein Bericht vorhanden. Bitte Ausgabe aus Schritt 4 schicken.')
else:
    rows=json.loads(p.read_text())
    frame=pd.DataFrame(rows)
    columns=['pose_method','detector','pitcher_tags','cameras','status','pitcher_valid_trials','min_pitcher_cameras_seen',
             'max_spout_tip_mm','max_pitcher_angle_deg',
             'max_relative_spout_to_cup_mm','exit_code']
    display(frame[columns])
    print('PASS erfordert alle Messungen, mindestens 2 Kameras sowie 0,5 mm/1°.')
    print('Der relative Spitze-zu-Tasse-Fehler wird separat ausgewiesen.')
    for variant in ('bundle_adjust_pupil_1tag_4cam',
                    'caliscope_pupil_1tag_4cam',
                    'caliscope_pupil_2tag_4cam'):
        image_path=folder/variant/'rgbd_montage.jpg'
        if image_path.exists():
            print('Kameraübersicht:',variant)
            display(Image(filename=str(image_path)))
    archive=Path(repo)/'runs/rig_comparison_results.zip'
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for file in folder.rglob('*'):
            if file.is_file() and file.suffix!='.npy':
                z.write(file,file.relative_to(Path(repo)/'runs'))
    print('Messberichte und Bilder herunterladen:')
    display(FileLink(str(archive)))


### Was wird hier tatsächlich getestet?

Bei **einem Tag** stehen alle Kameras auf einer Seite; bei **zwei Tags** sind sie mit 2+1 oder 2+2 Ansichten auf beide Seiten verteilt.

Die Gegenüberstellung **Caliscope gegen 2D-Optimierung** ist ein Vergleich der Positionsrekonstruktion aus denselben erkannten Tag-Ecken, **keine Gegenüberstellung zweier Kamera-Kalibrierungsprogramme**. Caliscope trianguliert Bildpunkte in 3D; unsere Anbindung errechnet daraus die Körperpose. Die bestehende Methode optimiert die 6D-Körperpose direkt aus allen 2D-Bildecken und nutzt zusätzlich virtuelle Tiefendaten als weiche Nebenbedingung. Sie unterscheiden sich also auch im Umfang der Sensordaten. Für einen späteren isolierten Algorithmusvergleich müssen wir diese Eingaben noch angleichen.

Die gemessenen Bilder stammen aus einer idealisierten MuJoCo-Szene mit exakt bekannter Kameraposition. Weitere Tests mit echten Kalibrierungsfehlern, Rauschen, Verdeckung und zeitlichen Bewegungen sind notwendig. Alle Details und Fehlerprotokolle landen in der ZIP-Datei.
